In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
BATCH_SIZE = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE = 1e-3
EPOCHS = 10
NOISE_FACTOR = 0.25

In [ ]:
train_data = datasets.CIFAR10(root='./data', train=True, transform=transforms.ToTensor(), download=True)
test_data = datasets.CIFAR10(root='./data', train=False, transform=transforms.ToTensor(), download=True)

train_dataloader = DataLoader(dataset= train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(dataset=test_data, batch_size=10, shuffle=False, pin_memory=True)

In [ ]:
train_data.class_to_idx

In [ ]:
train_data[0][0].shape

In [ ]:
class UNET_Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        # [3, 32, 32] -> [16, 16, 16]
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=(3, 3), stride=2, padding=1),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(inplace=True)
        )

        # [16, 16, 16] -> [32, 8, 8]
        self.enc2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), stride=2, padding=1),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(inplace=True)
        )

        # [32, 8, 8] -> [64, 4, 4]
        self.enc3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), stride=2, padding=1),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True)
        )

        # Bottleneck
        # [64, 4, 4] -> [128, 2, 2]
        self.bottleneck = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3, 3), stride=2, padding=1),
            nn.BatchNorm2d(num_features=128),
            nn.ReLU()
        )

        # Decoder
        self.up1 = nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=(3, 3), stride=2, padding=1, output_padding=1)

        self.dec1 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=64, kernel_size=(3,3), stride=1, padding=1),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True)
        )


        self.up2 = nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=(3, 3), stride=2, padding=1, output_padding=1)
        
        self.dec2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=32, kernel_size=(3, 3), stride=1, padding=1),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(inplace=True)
        )


        self.up3 = nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=(3, 3), stride=2, padding=1, output_padding=1)

        self.dec3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=16, kernel_size=(3, 3), stride=1, padding=1),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(inplace=True)
        )

        self.final_up = nn.ConvTranspose2d(in_channels=16, out_channels=3, kernel_size=(3, 3), stride=2, padding=1, output_padding=1)
        self.final_act = nn.Sigmoid()

    def forward(self, x):
        e1 = self.enc1(x)       # [batch, 16, 16, 16]
        e2 = self.enc2(e1)      # [batch, 32, 8, 8]
        e3 = self.enc3(e2)      # [batch, 64, 4, 4]

        b = self.bottleneck(e3)     # [batch, 128, 2, 2]

        d1 = self.up1(b)
        d1 = torch.cat([d1, e3], dim=1)     # Connect on channels: 64+64=128
        d1 = self.dec1(d1)

        d2 = self.up2(d1)
        d2 = torch.cat([d2, e2], dim=1)     # Connect on channels: 32+32=64
        d2 = self.dec2(d2)

        d3 = self.up3(d2)
        d3 = torch.cat([d3, e1], dim=1)     # Connect on channels: 16+16=32
        d3 = self.dec3(d3)

        out = self.final_up(d3)
        out = self.final_act(out)

        return out



In [ ]:
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.mse = nn.MSELoss()
        self.l1 = nn.L1Loss()
        self.alpha = alpha      # balance weight

    def forward(self, prediction, targets):
        mse_loss = self.mse(prediction, targets)
        l1_loss = self.l1(prediction, targets)

        total_loss = self.alpha * mse_loss + (1-self.alpha) * l1_loss

        return total_loss

In [ ]:
from pytorch_msssim import SSIM

class L1SSIMLoss(nn.Module):
    def __init__(self, alpha=0.84):
        super().__init__()
        self.ssim = SSIM(data_range=1.0, size_average=True, channel=3)
        self.l1 = nn.L1Loss()
        self.alpha = alpha

    def forward(self, predictions, targets):
        ssim_loss = 1 - self.ssim(predictions, targets)
        l1_loss = self.l1(predictions, targets)
        
        return self.alpha * ssim_loss + (1 - self.alpha) * l1_loss

In [ ]:
model = UNET_Autoencoder()
model.to(device=DEVICE)

criterion = L1SSIMLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

In [ ]:
model.train()
for epoch in range(EPOCHS):
    train_loss = 0.0
    for images, _ in train_dataloader:
        images = images.to(device=DEVICE)
        noisy_images = images + NOISE_FACTOR * torch.randn_like(images)
        noisy_images = torch.clamp(noisy_images, 0., 1.)

        outputs = model(noisy_images)
        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch: {epoch+1}/{EPOCHS}, Loss: {train_loss}")

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    dataiter = iter(test_dataloader)
    images, _ = next(dataiter)

    images = images.to(device=DEVICE)

    noisy_images = images + NOISE_FACTOR * torch.randn_like(images)
    noisy_images = torch.clamp(noisy_images, 0., 1.)

    outputs = model(noisy_images)

    images = images.cpu().numpy()
    noisy_images = noisy_images.cpu().numpy()
    outputs = outputs.cpu().numpy()

    fig, axes = plt.subplots(nrows=3, ncols=10, figsize=(12, 6))

    for i in range(10):
        orig_img = images[i].transpose(1, 2, 0)
        noisy_img = noisy_images[i].transpose(1, 2, 0)
        denoised_img = outputs[i].transpose(1, 2, 0)

        axes[0, i].imshow(orig_img)
        axes[0, i].axis("off")
        axes[0, i].set_title("Original")

        axes[1, i].imshow(noisy_img)
        axes[1, i].axis("off")
        axes[1, i].set_title("Noisy")

        axes[2, i].imshow(denoised_img)
        axes[2, i].axis("off")
        axes[2, i].set_title("Denoised")

    plt.tight_layout()
    plt.show()
